# KrishiDrishti — Satellite-Based Crop Mapping, Moisture-Stress Detection & Irrigation Advisory

**A full-stack AI + remote-sensing system: draw a box anywhere in northern India → per-field crop map, growth-stage-aware moisture stress, 8-day crop-water deficit, and an irrigation advisory — in ~2–3 minutes.**

*Built for Bharatiya Antariksh Hackathon (BAH) (an initiative by ISRO) Problem Statement 6 (AI-driven crop type, moisture stress detection & irrigation advisory using optical + microwave satellite data), and extended into a deployable web service.*

> Author: Harshul Aggarwal · [LinkedIn / GitHub links] · [Date]

---

## 1. Project Overview

Irrigation timing is one of the highest-leverage decisions in Indian agriculture, yet field-level guidance rarely exists: crop maps are coarse or stale, stress detection ignores the crop's growth stage, and water-deficit numbers are seldom translated into an actionable "irrigate now / wait" call.

This project builds that missing chain end-to-end, from raw satellite pixels to a per-field advisory, and wraps it in a web application:

1. **Crop-type classification** — an 8-class per-field crop map (Wheat, Mustard, Lentil, No-crop/Fallow, Sugarcane, Maize, Rice, Other) from a full season of Sentinel-1 SAR + Sentinel-2 optical time series.
2. **Moisture-stress detection** — growth-stage-aware stress levels per field per 10-day dekad, fusing a physics-based water balance with observed spectral/SAR stress indicators.
3. **8-day crop-water deficit + irrigation advisory** — FAO-56 water-balance deficit in mm and m³ per 8-day period, translated into a 5-level advisory (Out of season / No stress / Watch / Irrigation advised / Severe).
4. **A web service** — users draw an area of interest (AOI) on a map; the backend fetches that season's imagery from Google Earth Engine, runs the full 11-stage pipeline, and returns an interactive dashboard with color-coded field-level maps and downloadable GeoTIFF/CSV products.

**Headline numbers**

| Metric | Value |
|---|---|
| Crop classification (test) | **OA 0.78** · macro-F1 0.55 · Cohen's κ 0.68 |
| Honest generalization (5-fold chip-level CV) | **OA ~0.73** |
| Sowing-date detection (NDVI green-up) | **87%** of fields observed directly |
| LSTM stress emulator vs FAO-56 Ks (held-out chips) | **R² 0.96** · MAE 0.036 · stress-flag P 0.94 / R 0.95 |
| End-to-end AOI job (5×5 km, incl. GEE fetch) | **~140–160 s** |
| Demonstrator scale (UP pilot) | 537 chips · 2,992 fields · 18 dekads · 1.3 Mm³ season deficit over 793 ha |

Every detection step is **causal** (no future dekads used), so the same pipeline can run mid-season on a live crop, not just retrospectively.

---

## 2. Dataset Summary

| Source | Content | Role |
|---|---|---|
| **AgriFieldNet India 2021-22** (Radiant Earth / Zindi) | 5,771 labeled fields, 13 raw crop classes, 4 pilot regions; avg field ≈ 0.26 ha (~26 px at 10 m) | Ground truth for training/validation |
| **Sentinel-2 L2A** (via Google Earth Engine) | 11 monthly composites Jun 2021–Apr 2022; 9 bands (B02–B08, B8A, B11); s2cloudless + SCL cloud masking, median composite | Optical vegetation signal (NDVI, NDMI, red-edge indices, EVI, gNDWI + raw bands) |
| **Sentinel-1 IW GRD** | 11 monthly VV/VH backscatter composites (dB) | All-weather structure/moisture signal; region-normalized |
| **Dekadal "SEASON" composites** (B04/B05/B08) | 18 ten-day composites Nov–Apr | Fine-grained phenology: green-up sowing detection, dekadal NDVI/NDRE |
| **NASA POWER** daily meteorology | T, RH, wind, solar, rain on a 0.5° grid | FAO-56 Penman-Monteith ET₀ for the water balance |

**Extracted training corpus:** 1,151 chips of 256×256 px at 10 m (T=11 months), across Uttar Pradesh (537), Rajasthan (238), Odisha (200) and Bihar (176). Splits are **chip-level stratified 80/10/10** (stratum = region × rarest-present crop) so spatially-correlated pixels never leak across train/test.

**Class consolidation:** 13 raw crops → 8 classes; minor crops (green pea, garlic, gram, coriander, …) fold into "Other". Rice remains its own class.

---

## 3. Workflow

```mermaid
flowchart TD
    A[User draws AOI + season year<br/>React + MapLibre frontend] --> B[FastAPI backend<br/>validate zone & size, dedupe, queue]
    B --> C[GEE fetch via parallel computePixels<br/>11 monthly S2 + 11 monthly S1 + 18 dekadal composites]
    C --> D[Tile into 256x256 chips<br/>Felzenszwalb pseudo-field segmentation]
    D --> E[1,305-dim per-field features<br/>spectral indices + SAR + raw bands x 5 field stats]
    E --> F[LightGBM 8-class classifier<br/>+ TempCNN rice-rescue]
    F --> G[Per-field crop map]
    C --> H[NASA POWER weather<br/>FAO-56 Penman-Monteith ET0]
    D --> I[Dekadal NDVI/NDRE + monthly NDMI, VV/VH<br/>VCI, NDVI-z, soil-moisture proxy]
    I --> J[Sowing dates from NDVI green-up<br/>kharif-trough aware, causal]
    G --> K[Stage-aware FAO-56 water balance<br/>per weather-cell x crop x sowing-bin]
    H --> K
    J --> K
    K --> L[8-day crop-water deficit<br/>mm and m3 per field]
    K --> M[Rule-based advisory fusion<br/>5 levels, physics x spectral cross-check]
    I --> M
    I --> N[LSTM Ks emulator<br/>satellite-only stress, weather-free fallback]
    K --> N
    L --> O[Deliverables]
    M --> O
    N --> O
    O --> P[Interactive dashboard + field-level crop & advisory maps<br/>GeoTIFFs, CSVs, time-series drill-downs]
```

---

## 4. Tech Stack

**Front-end**
- **React 18 + Vite** — single-page app
- **MapLibre GL JS** — satellite basemap, drag-to-draw AOI, supported-zone mask, pilot-area badges (VALIDATED vs EXPERIMENTAL)
- Live 11-stage progress tracking (2 s polling), results panel with KPIs, crop-mix chart, deficit sparkline, file downloads

**Back-end**
- **FastAPI + Uvicorn** — job API (`POST /api/jobs`, per-stage progress, results/dashboard/file serving)
- **Google Earth Engine Python API** — server-side compositing; ~40 parallel `computePixels` requests fetch a full season for a 10×10 km AOI in **~40 s**
- Production hygiene: job dedupe by (bbox, year) hash with instant cache hits, one active job per client, submission rate-limiting, global concurrency semaphore, resumable pipeline stages, path-traversal-guarded file serving

**ML / geospatial**
- **LightGBM** (classifier) · **PyTorch** (TempCNN, LSTM) · **scikit-learn** / **scikit-image** (metrics, Felzenszwalb segmentation)
- **rasterio** (GeoTIFF I/O, georeferencing) · **NumPy** · **matplotlib / Pillow** (map rendering)
- Self-contained HTML dashboard (inline SVG charts, no framework) served by the API or opened as a local file

---

## 5. Model & System Architecture

### 5.1 Crop classifier — LightGBM on engineered time-series features

- **Features (1,305-dim per field):** 261 per-pixel features × 5 field statistics (mean/std/median/min/max). Per-pixel blocks: NDVI/NDMI monthly series, cloud-mask fractions, region-normalized VV/VH/ratio SAR series, red-edge indices (NDRE, CIre, RE-NDVI), EVI + gNDWI, **99 raw S2 band-month values** (the single biggest accuracy lever found), kharif SAR-flooding cues, and static context.
- **Model:** LightGBM, 500 trees, 31 leaves, lr 0.05, class-balanced weights.
- **Rice-rescue specialist:** a 13-class **TempCNN** (temporal CNN over the 7-channel monthly sequence + 8 static features) re-examines fields the LGB called "Fallow" and flips them to Rice when P(rice) > 0.70 — recovering a rare class the tabular model under-predicts.
- **Validation discipline:** chip-level splits (no spatial leakage); the reported **CV OA ~0.73** is the honest generalization number. Error analysis shows the ceiling is dominated by Wheat↔Mustard spectral overlap at 10 m on 0.26 ha smallholder fields — a data limitation, not a tuning one (confirmed across RF/XGB/deep baselines, texture, phenology and ensembling experiments).

### 5.2 Sowing-date detection — per-field phenology

NDVI green-up crossing (threshold 0.30) on the dekadal series, **kharif-trough aware** (a field already green in November must first dip and re-cross, so standing kharif crop isn't mistaken for rabi sowing), minus a crop-specific emergence lag. 87% of fields observed directly; the rest fall back to crop-median or literature calendars. Wheat sowing spread (p10–p90: Oct 20 → Dec 13) shifts a field's irrigation requirement by up to ~3×, which is why per-field timing matters.

### 5.3 Water balance & advisory — physics first, ML where physics can't reach

- **FAO-56 dual-run water balance** per (weather cell × crop × sowing bin): stage-dependent Kc curves, root-zone bucket, depletion vs readily-available-water (RAW) trigger. A *scheduled* run yields the advised irrigation calendar; a *rainfed* run yields the stress coefficient **Ks** and the **8-day deficit** (ETc − ETa) demanded by PS-6.
- **Advisory fusion — fixed, auditable rules (deliberately not a learned model):** "Irrigation advised" fires when the bucket hits the RAW trigger in a water-sensitive stage, or when moderate spectral stress (VCI / NDVI-z / NDMI-z) coincides with a dry Sentinel-1 soil-moisture proxy or Ks < 0.85; "Severe" requires Ks < 0.85 **and** strong spectral stress in a sensitive stage; "Watch" at 80% of trigger depletion or mild spectral stress. Physics and observation must corroborate each other before a farmer is told to spend water.
- **LSTM Ks emulator (the satellite-only fallback):** a 2-layer causal LSTM (hidden 64, 19 spectral/SAR input features, **no weather inputs**) distilled against the FAO-56 rainfed Ks. Held-out chips: MAE 0.036, R² 0.96, stress-flag precision 0.94 / recall 0.95. Its role is stress estimation where meteorological data is missing or late; the dashboard automatically flags (and the advisory ignores it) when its agreement with the water balance is poor on out-of-distribution areas/seasons.

### 5.4 Serving arbitrary AOIs

The training pipeline was parameterized end-to-end via environment variables, so the identical code serves any bbox: GEE composites are built with the **same compositing recipe as the training tiles** (verified band-for-band), pseudo-fields come from Felzenszwalb segmentation on peak-NDVI (cadastral boundaries don't exist for arbitrary AOIs), and a northern-India gate (68–89°E, 19–32.5°N) blocks requests where the model has zero training support. The four pilot areas are badged **VALIDATED**; everywhere else is explicitly **EXPERIMENTAL**.

---

## 6. Results & Deliverables

| Deliverable | Form |
|---|---|
| Crop-type map | Field-level 10 m mosaic PNG + per-chip GeoTIFFs with colormap |
| Stage-wise moisture-stress / advisory maps | 18 per-dekad translucent overlays + GeoTIFFs |
| 8-day water deficit | Per-field mm & m³ series, CSVs, 23-band deficit GeoTIFFs |
| Irrigation advisory | 5-level per-field per-dekad, CSV + dashboard ribbon |
| Dashboard | Self-contained HTML: KPIs, season timeline, demand-vs-rainfall, sowing histograms, per-field drill-downs |

*(Insert 2–3 screenshots here: the draw-AOI web UI, the field-level crop map, the advisory dashboard.)*

---

## 7. Key Engineering Lessons

1. **Chip-level splits or nothing** — pixel/field-level splits inflate accuracy through spatial autocorrelation; the honest CV number (0.73) cost several "improvements" that evaporated under proper validation.
2. **Raw bands beat clever indices** — after 20+ engineered-feature experiments (GLCM texture, phenology metrics, flowering indices, SAR entropy…), the only material win was feeding raw band-month values alongside the indices.
3. **Physics + ML beats either alone** — the FAO-56 balance provides an interpretable, defensible advisory; the LSTM extends it to weather-sparse operation; fixed fusion rules keep the farmer-facing output auditable.
4. **Latency lives in the slowest composite** — GEE `computePixels` wall-time is bounded by the slowest of ~40 parallel requests (~33 s), not AOI area, until the per-request payload cap forces tiling.
5. **Honesty is a feature** — the UI labels unvalidated regions EXPERIMENTAL and the dashboard self-flags when the ML emulator disagrees with the physics; trust in an advisory product is earned by admitting limits.

---

## 8. Future Work

- GEE **service-account** auth + cloud deployment (currently runs on personal credentials)
- Cross-job composite cache keyed (bbox, month, product)
- Field-delineation model to replace pseudo-field segmentation
- Finer weather (IMD 0.25° / ERA5-Land 9 km); canal command-area rollups of per-field volumes
- Live mid-season operation (all steps already causal) and a second validated region
- Breaking the 0.73 OA wall: new labeled wheat/mustard data or hyperspectral (EnMAP) for a recent season

---

## 9. References

1. Allen, R.G., Pereira, L.S., Raes, D., Smith, M. (1998). *Crop Evapotranspiration — Guidelines for Computing Crop Water Requirements.* FAO Irrigation & Drainage Paper 56.
2. Radiant Earth Foundation & IDinsight (2022). *AgriFieldNet India Competition Dataset: Crop Types in Ground Reference Data.* Radiant MLHub.
3. Gorelick, N. et al. (2017). *Google Earth Engine: Planetary-scale geospatial analysis for everyone.* Remote Sensing of Environment, 202.
4. Ke, G. et al. (2017). *LightGBM: A Highly Efficient Gradient Boosting Decision Tree.* NeurIPS 30.
5. Pelletier, C., Webb, G.I., Petitjean, F. (2019). *Temporal Convolutional Neural Network for the Classification of Satellite Image Time Series.* Remote Sensing, 11(5).
6. Hochreiter, S., Schmidhuber, J. (1997). *Long Short-Term Memory.* Neural Computation, 9(8).
7. Felzenszwalb, P.F., Huttenlocher, D.P. (2004). *Efficient Graph-Based Image Segmentation.* IJCV, 59(2).
8. Kogan, F.N. (1995). *Application of vegetation index and brightness temperature for drought detection.* Advances in Space Research, 15(11). (VCI)
9. ESA Copernicus Sentinel-1 & Sentinel-2 mission documentation; Sen2Cor / s2cloudless cloud masking.
10. NASA POWER Project — agroclimatology daily data, langley.nasa.gov/power.

---

*Stack: Python · PyTorch · LightGBM · Google Earth Engine · FastAPI · React · MapLibre GL · rasterio*
